<a href="https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/02_text_and_phoneme_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Text and phoneme preparation
**Goal:** turn LJSpeech transcripts into clean text that MFA can align, and fix the *phoneme inventory* used by the whole project.

**Why this matters**
- MFA looks every word up in a pronunciation dictionary. Digits, accents, brackets and odd punctuation cause out-of-vocabulary words, and those make bad or failed alignments.
- Punctuation is useful later (a comma often means a pause), so we keep it as a *side-channel* (`puncts`) while giving MFA only clean words.
- One phoneme inventory file (`src/features/phones.py`) is imported by Phase 1 *and* Phase 2, so the symbols can never drift apart.

**This notebook answers:** what text normalization is, why we strip punctuation for alignment but keep it as a feature, and what ARPAbet stress digits mean (AH0 / AH1 / AH2 = no / primary / secondary stress).

**Reads:** LJSpeech `metadata.csv`. **Writes:** `data/processed/transcripts_clean.csv`, MFA corpora in `/content`, `src/features/phones.py`.

In [2]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [3]:
# ---- Make sure the dataset exists on Colab's local disk (it is wiped when the session restarts)
import subprocess
if not os.path.isdir(RAW):
    archive = os.path.join(PROJECT_ROOT, 'data/raw/LJSpeech-1.1.tar.bz2')
    assert os.path.exists(archive), f'Archive not found: {archive}'
    subprocess.run(['tar', '-xjf', archive, '-C', os.path.dirname(RAW)], check=True)
print('wavs:', len(os.listdir(os.path.join(RAW, 'wavs'))), '(expect 13100)')

wavs: 13100 (expect 13100)


In [4]:
# ---- Load metadata
import pandas as pd, csv
meta = pd.read_csv(os.path.join(RAW, 'metadata.csv'), sep='|', header=None,
                   names=['id', 'raw_text', 'norm_text'], quoting=csv.QUOTE_NONE, encoding='utf-8')
meta['norm_text'] = meta['norm_text'].fillna(meta['raw_text'])
print(len(meta), 'clips')
meta.head(3)

13100 clips


,id,raw_text,norm_text
0,LJ001-0001,"Printing, in the only sense with which we are ...","Printing, in the only sense with which we are ..."
1,LJ001-0002,in being comparatively modern.,in being comparatively modern.
2,LJ001-0003,For although the Chinese took impressions from...,For although the Chinese took impressions from...


## Step 1: Tokenize and clean
For each word we keep: the cleaned lowercase word (for MFA) and a punctuation code `n` (none), `c` (comma/semicolon/colon) or `p` (. ! ?) that followed it in the original text.
Hyphens become spaces ("well-known" -> "well known") because the pieces are far more likely to be in the dictionary. Accents are stripped (é -> e).

In [5]:
import re, unicodedata

def tokenize(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    words, puncts = [], []
    for tok in text.split():
        m = re.search(r'([,;:.!?])["\u201d\u2019\')\]]*$', tok)
        code_ = 'n' if not m else ('c' if m.group(1) in ',;:' else 'p')
        core = re.sub(r"[^a-z'\-]", ' ', tok.lower()).replace('-', ' ')
        parts = [p.strip("'") for p in core.split()]
        parts = [p for p in parts if p]
        for i, p in enumerate(parts):
            words.append(p)
            puncts.append(code_ if i == len(parts) - 1 else 'n')
    return words, puncts

print(tokenize('Mr. Smith’s well-known “café”, however, [sic] left.'))

(['mr', "smith's", 'well', 'known', 'cafe', 'however', 'sic', 'left'], ['p', 'n', 'n', 'n', 'c', 'c', 'n', 'p'])


In [6]:
toks = meta['norm_text'].apply(tokenize)
meta['mfa_text'] = toks.apply(lambda t: ' '.join(t[0]))
meta['puncts'] = toks.apply(lambda t: ','.join(t[1]))
meta['n_words'] = toks.apply(lambda t: len(t[0]))
assert (meta['mfa_text'].str.len() > 0).all(), 'some transcripts became empty'
meta[['id', 'mfa_text', 'puncts', 'n_words']].to_csv(os.path.join(PROCESSED, 'transcripts_clean.csv'), index=False)
print('saved. words/clip: mean %.1f, max %d' % (meta.n_words.mean(), meta.n_words.max()))
meta[['norm_text', 'mfa_text']].sample(5, random_state=1).to_dict('records')

saved. words/clip: mean 17.2, max 40


[{'norm_text': '(three) evidence establishing the ownership of the murder weapon,',
  'mfa_text': 'three evidence establishing the ownership of the murder weapon'},
 {'norm_text': 'These men managed to cut a hole in the ceiling of the ward near the iron cage on the landing, and so got access to the roof.',
  'mfa_text': 'these men managed to cut a hole in the ceiling of the ward near the iron cage on the landing and so got access to the roof'},
 {'norm_text': 'When visited one day in the condemned cell, just as St. Sepulchre\'s clock was striking, he looked up and said laughingly, "Go along, clock;',
  'mfa_text': "when visited one day in the condemned cell just as st sepulchre's clock was striking he looked up and said laughingly go along clock"},
 {'norm_text': 'Frazier testified before the Commission that Oswald carried no lunch sack that day.',
  'mfa_text': 'frazier testified before the commission that oswald carried no lunch sack that day'},
 {'norm_text': 'They should be persons

In [7]:
# ---- Vocabulary check: what will MFA's dictionary have to cover?
from collections import Counter
vocab = Counter(w for t in meta['mfa_text'] for w in t.split())
chars = Counter(''.join(meta['mfa_text']))
print('vocabulary:', len(vocab), '| singletons:', sum(1 for n in vocab.values() if n == 1))
print('characters left:', sorted(chars))
print('top words:', vocab.most_common(10))

vocabulary: 14058 | singletons: 5513
characters left: [' ', "'", 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
top words: [('the', 18357), ('of', 8729), ('and', 6480), ('to', 6318), ('in', 4826), ('a', 4424), ('was', 3811), ('that', 2994), ('he', 2730), ('his', 2032)]


## Step 2: Build the MFA corpora
MFA wants a folder where each `x.wav` has a `x.lab` with its transcript. We **symlink** the wavs (no copying) and write the `.lab` files.
We build a *small* corpus (first 200 clips) to test the pipeline quickly, and the *full* one.

In [8]:
def build_corpus(ids, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    text = dict(zip(meta['id'], meta['mfa_text']))
    for cid in ids:
        dst = os.path.join(out_dir, cid + '.wav')
        if not os.path.exists(dst):
            os.symlink(os.path.join(RAW, 'wavs', cid + '.wav'), dst)
        with open(os.path.join(out_dir, cid + '.lab'), 'w') as f:
            f.write(text[cid])
    print(out_dir, ':', len(ids), 'utterances')

ids = meta['id'].tolist()
build_corpus(ids[:200], '/content/mfa_corpus_small')
build_corpus(ids, '/content/mfa_corpus')

/content/mfa_corpus_small : 200 utterances
/content/mfa_corpus : 13100 utterances


## Step 3: The shared phoneme inventory
This writes `src/features/phones.py` (ARPAbet used by MFA's `english_us_arpa`): 15 vowels, 24 consonants, plus `sil` (silence) and `spn` (unknown-word noise).
Phones are grouped into classes (vowel, stop, fricative, ...) because durations behave very differently per class, which you will see in notebook 04.

In [9]:
# ---- Write shared module: phoneme inventory
import os, importlib
_src = r'''"""Shared phoneme inventory (ARPAbet, as used by MFA english_us_arpa). Used by Phase 1 and Phase 2."""
import re

VOWELS = ['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'EH', 'ER', 'EY', 'IH', 'IY', 'OW', 'OY', 'UH', 'UW']
DIPHTHONGS = ['AW', 'AY', 'EY', 'OW', 'OY']
CONSONANT_CLASS = {
    'stop': ['B', 'D', 'G', 'K', 'P', 'T'],
    'affricate': ['CH', 'JH'],
    'fricative': ['DH', 'F', 'HH', 'S', 'SH', 'TH', 'V', 'Z', 'ZH'],
    'nasal': ['M', 'N', 'NG'],
    'liquid': ['L', 'R'],
    'glide': ['W', 'Y'],
}
SPECIAL = ['sil', 'spn']          # silence, unknown-word noise
PAD = '<pad>'

BASE_PHONES = VOWELS + [p for v in CONSONANT_CLASS.values() for p in v]
PHONE_LIST = [PAD] + sorted(BASE_PHONES) + SPECIAL
PHONE2ID = {p: i for i, p in enumerate(PHONE_LIST)}

_CLASS = {p: 'vowel' for p in VOWELS}
for _c, _ps in CONSONANT_CLASS.items():
    for _p in _ps:
        _CLASS[_p] = _c
_CLASS['sil'] = 'sil'
_CLASS['spn'] = 'spn'
CLASS_LIST = ['vowel', 'stop', 'affricate', 'fricative', 'nasal', 'liquid', 'glide', 'sil', 'spn']
CLASS2ID = {c: i for i, c in enumerate(CLASS_LIST)}


def split_stress(label):
    """'AH0' -> ('AH', 0); 'T' -> ('T', -1); '' -> ('sil', -1); 'spn' -> ('spn', -1)."""
    if label == '' or label is None:
        return 'sil', -1
    if label.lower() in ('sil', 'sp'):
        return 'sil', -1
    if label.lower() == 'spn':
        return 'spn', -1
    m = re.match(r'^([A-Z]+?)([012])?$', label)
    if not m:
        return label, -1
    return m.group(1), (int(m.group(2)) if m.group(2) is not None else -1)


def phone_class(base):
    return _CLASS.get(base, 'unknown')
'''
_p = os.path.join(PROJECT_ROOT, "src/features/phones.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/features/phones.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/features/phones.py


In [10]:
from src.features.phones import *
print(len(BASE_PHONES), 'base phones:', sorted(BASE_PHONES))
print(split_stress('AH0'), split_stress('T'), split_stress(''), split_stress('spn'))
print({p: phone_class(p) for p in ['IY', 'T', 'S', 'M', 'L', 'W', 'sil']})

39 base phones: ['AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH', 'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N', 'NG', 'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W', 'Y', 'Z', 'ZH']
('AH', 0) ('T', -1) ('sil', -1) ('spn', -1)
{'IY': 'vowel', 'T': 'stop', 'S': 'fricative', 'M': 'nasal', 'L': 'liquid', 'W': 'glide', 'sil': 'sil'}
